In [1]:
import pandas as pd
import geopandas as gpd
import os
import dotenv
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
trees = pd.read_csv('data/LSM_Trees.csv')
geo_trees = gpd.read_file('data/LSM_Trees.geojson')
station_walkshed = gpd.read_file('data/station_walksheds.geojson')
print(geo_trees.shape)
trees.drop(columns=['OBJECTID', 'Active'], inplace=True)
trees.head(20)

(191292, 7)


,X,Y,Tree_ID,Common,DBH,Location
0,1.453890e+06,532845.493557,131013,OAK WILLOW,13.0,821 HARVARD PL
1,1.456410e+06,530778.584305,129548,CRAPEMYRTLE SPP,3.0,653 LLEWELLYN PL
2,1.455563e+06,530469.294569,129092,MAHONIA LEATHERLEAF,1.0,1142 BOLLING RD
3,1.456795e+06,533546.845829,129706,OAK WILLOW,27.0,227 COLVILLE RD
4,1.457469e+06,533337.790144,129672,PRIVET SPP,2.0,2511 MONTROSE CT
5,1.454012e+06,535155.771089,130313,CHERRY YOSHINO,9.0,530 QUEENS RD
6,1.455674e+06,534958.936180,128899,SERVICEBERRY SPP,3.0,218 CIRCLE AV
7,1.456057e+06,534587.450703,128931,OAK WILLOW,19.0,2205 CRESCENT AV
8,1.456359e+06,528558.422735,130004,MAPLE RED,11.0,1420 SCOTLAND AV
9,1.453142e+06,532640.889635,130888,CHERRY YOSHINO,16.0,1128 QUEENS RD


In [3]:
geo_trees = geo_trees.to_crs("EPSG:26917")
trees_filtered = gpd.sjoin(
    geo_trees,
    station_walkshed,
    predicate="within",
    how="inner"
)
print(trees_filtered.shape)

(5249, 11)


In [4]:
trees_filtered['Common'].value_counts()

Common
OAK: WILLOW          784
MAPLE: RED           610
OAK: LAUREL          492
OAK: SHUMARD         385
OAK: SAWTOOTH        365
                    ... 
CHERRY: BLACK          1
OAK: SOUTHERN RED      1
ASH: WHITE             1
ASH: GREEN             1
FIG: COMMOM            1
Name: count, Length: 116, dtype: int64

In [5]:
tree_types = trees_filtered['Common'].unique().tolist()

In [6]:
def get_info_from_api(item, client, model="gpt-4o-mini", temperature=0.0):
    prompt = f"""
    For the following tree, determine if it is deciduous. If it is, return 1, if it is not, return 0. ONLY RETRUN A 1 OR 0:
    {item}
    """

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert in tree identification and classification."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

In [7]:
results = {}

for item in tree_types:
    api_output = get_info_from_api(item, client)
    results[item] = api_output
    print(f"{item} → {api_output}")

MAPLE SUGAR → 1
OAK WILLOW → 1
MAGNOLIA SAUCER → 1
PHOTINIA SPP → 1
ZELKOVA JAPANESE → 1
CRAPEMYRTLE SPP → 1
MAGNOLIA CUCUMBER TREE → 1
OAK WATER → 1
MAPLE SILVER → 1
DOGWOOD FLOWERING → 1
ELM CHINESE → 1
HORNBEAM EUROPEAN → 1
ASH GREEN → 1
MAPLE RED → 1
PECAN → 1
OAK SOUTHERN RED → 1
REDCEDAR EASTERN → 0
ARBORVITAE AMERICAN → 0
CHERRY/PLUM/PEACH SPP → 1
SWEETGUM AMERICAN → 1
OAK NORTHERN RED → 1
PEAR CALLERY → 1
CHERRY BLACK → 1
OAK SCARLET → 1
STUMP → 0
CHERRY HIGAN → 1
MULBERRY WHITE → 1
CRABAPPLE SPP → 1
HORNBEAM AMERICAN → 1
MAPLE FREEMAN → 1
REDBUD EASTERN → 1
MAGNOLIA SOUTHERN → 1
CYPRESS LEYLAND → 0
WAXMYRTLE SOUTHERN → 0
CYPRESS ITALIAN → 0
OAK WHITE → 1
HACKBERRY SUGAR → 1
JUNIPER ROCKY MOUNTAIN → 0
ELM WINGED → 1
PRIVET GLOSSY → 1
HOLLY CHINESE → 0
CRAPE MYRTLE → 1
MAPLE PAINTED → 1
TULIPTREE → 1
Carolina Silverbell → 1
MAPLE: RED → 1
JAPANESE SNOWBELL → 1
BLACKGUM: COMMON → 1
HORNBEAM: AMERICAN → 1
CRAPEMYRTLE: SPP → 1
OAK: WILLOW → 1
OAK: SHUMARD → 1
DOGWOOD: FLOWERING → 1

In [8]:
trees_filtered["Deciduous"] = trees_filtered["Common"].map(results)
trees_filtered["Deciduous"].value_counts()
trees_filtered = trees_filtered.drop(columns=['index_right', 'name'])

In [9]:
trees_filtered.to_csv('data/trees_cleaned.csv', index=False)
trees_filtered.to_file('data/trees_cleaned.geojson', driver='GeoJSON')